In [ ]:
!zip -r outputs.zip outputs
from google.colab import files
files.download("outputs.zip")

In [ ]:
!pip install accelerate diffusers controlnet_aux

In [ ]:
import os
from pathlib import Path
from PIL import Image
import torch
from controlnet_aux import CannyDetector
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from diffusers.utils import load_image, make_image_grid

In [ ]:
class StyleTransferPipeline:
    IMAGES_NUMBER = 3
    INFERENCE_STEPS_NUMBER = 20
    MAX_SIDE_IMG_SIZE = 768
    def __init__(
        self,
        ip_scale: float, 
        control_net_scale: float, 
        guidance_scale: int, 
        base_model: str = "Yntec/AbsoluteReality",
        controlnet_model: str = "lllyasviel/sd-controlnet-canny",
        ip_adapter_repo: str = "h94/IP-Adapter",
        ip_adapter_weight: str = "ip-adapter_sd15.bin",
        output_dir: str = "outputs",
    ):
        
        # Hyperparameters
        self.ip_scale = ip_scale
        self.control_net_scale = control_net_scale
        self.guidance_scale = guidance_scale

        self.ip_adapter_repo = ip_adapter_repo
        self.ip_adapter_weight = ip_adapter_weight

        # Output dir
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        # Initialize the Canny detector and load the ControlNetModel
        self.canny = CannyDetector()
        controlnet = ControlNetModel.from_pretrained(controlnet_model,
                                             torch_dtype=torch.float16,
                                             use_safetensors=True)
        # create the stable diffusion pipeline with the selected model 
        self.pipeline = StableDiffusionControlNetPipeline.from_pretrained(base_model,
                                                                 controlnet=controlnet,
                                                                 torch_dtype=torch.float16)        
        self.pipeline.enable_model_cpu_offload()

    def resize_img(self, img: Image.Image, multiple_of: int = 8) -> Image.Image:
        w, h = img.size
        scale = min(1.0, self.MAX_SIDE_IMG_SIZE / max(w, h))
        new_w, new_h = round(w * scale / multiple_of) * multiple_of, round(h * scale / multiple_of) * multiple_of
        if (new_w, new_h) == (w, h):
            return img
        return img.resize((new_w, new_h), Image.LANCZOS)

    def load_ip_adapters(self, n: int):
        self.pipeline.load_ip_adapter(
            self.ip_adapter_repo, 
            subfolder="models", 
            weight_name=[self.ip_adapter_weight] * n)
        self.pipeline.enable_model_cpu_offload()


        
    def save_results(
        self,
        images: list[Image.Image],
        content_path: Path,
        style_paths: list[Path],
    ) -> list[Path]:
        
        saved = []
        for i, img in enumerate(images):
            content_stem = content_path.stem
            style_stems = "_".join(p.stem for p in style_paths)
            filename = (
                f"{content_stem}__{style_stems}__"
                f"ip{self.ip_scale}_cn{self.control_net_scale}_cfg{self.guidance_scale}_v{i + 1}.jpg"
            )
            out_path = str(self.output_dir / filename)
            img.save(out_path, quality=95)
            print(f"  saved → {out_path}")
            saved.append(out_path)   
        return saved
            
    def stylise(self,         
                content_path: str,
                style_paths: list, 
                prompt: str,
                ip_load: bool = True
                ):
        negative_prompt = "low quality, blurry, disorted, deformed"
        positive_prompt = prompt
        
        # load content (+ resize) and style image
        content_image = self.resize_img(load_image(str(content_path)))
        style_images = [
            load_image(str(s)) for s in style_paths
        ]

        if ip_load:
            self.load_ip_adapters(len(style_images))
        
        canny_img = self.canny(content_image, detect_resolution=min(content_image.size), image_resolution=min(content_image.size))
        w, h = content_image.size

        stylised_images = self.pipeline(
            prompt=positive_prompt,
            negative_prompt=negative_prompt,
            height=h,
            width=w,
            ip_adapter_image=style_images,
            image=canny_img,
            guidance_scale=self.guidance_scale,
            controlnet_conditioning_scale= self.control_net_scale,
            num_inference_steps=self.INFERENCE_STEPS_NUMBER,
            num_images_per_prompt=self.IMAGES_NUMBER
        ).images

        self.save_results(stylised_images, Path(content_path), [Path(s) for s in style_paths])
        return make_image_grid(stylised_images, cols=self.IMAGES_NUMBER, rows=1)
        


# Inference

PARAMS:

**- guidance_scale:** [5 - 10] how strong to follow the text prompt;<br>
**- control_net_scale:** [0.5 - 1.0] lower scale if too much content leaking. If losing too much of the original structure - raise; <br>
**- ip_scale:** [0.4 - 0.8] lower value - prompt has more influence; higher value - style image has more influence. <br>

In [ ]:
# Change here
ip_scale = 0.5
control_net_scale = 0.7
guidance_scale = 6

In [ ]:
pipeline = StyleTransferPipeline(
    ip_scale = ip_scale,  
    control_net_scale = control_net_scale,  
    guidance_scale = guidance_scale
)

In [ ]:
content_path = "puppy.jpg"
style_paths =["starry_night.jpg", "circles_paint.jpg"]
prompt="cute puppy, high quality, masterpiece"

In [ ]:
results = pipeline.stylise(    
    content_path = content_path,
    style_paths = style_paths,
    prompt = prompt,
    ip_load = True # change to False if already ran 1 time, by running the second time the number of style images didnt change
)
results